# Parity-Net Colab Runner
This notebook clones the GitHub repo, trains the mean-field-style residual network from `MOTIVATION.md`, saves checkpoints to Google Drive, and runs PCA rank-reduction analysis.
Before running, set `GITHUB_REPO_URL` to the URL of your pushed repo.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
REPO_DIR = "/content/parity"
DRIVE_RUN_DIR = "/content/drive/MyDrive/parity_runs/example_run"
!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!pip install -e .


## Create Configs
`barrier_c` is in the training config because it enters as a loss regularizer. If left as `None`, the training code uses `7 / N`.


In [ ]:
from pathlib import Path
import yaml
config = {
    "model": {
        "input_dim": 32,
        "relevant_dim": 16,
        "N": 1024,
        "L": 4,
        "activation": "silu",
        "use_readout_barrier": True,
        "hidden_weight_variance": 1.0,
        "readout_weight_variance": 1e-4,
        "bias": False,
    },
    "training": {
        "train_samples": 100_000,
        "test_samples": 20_000,
        "batch_size": 512,
        "epochs": 20,
        "seed": 0,
        "device": "auto",
        "dtype": "float32",
        "log_every": 100,
        "checkpoint_every": 1,
        "output_dir": DRIVE_RUN_DIR,
        "barrier_c": None,
        "barrier_lambda": 10.0,
        "optimizer": {
            "name": "adamw",
            "lr": 1e-3,
            "weight_decay": 0.0,
            "momentum": 0.0,
            "betas": [0.9, 0.999],
        },
    },
}
config_path = Path(DRIVE_RUN_DIR) / "config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
config_path


## Train
This runs the repo command and saves intermediate checkpoints to Google Drive.


In [ ]:
!parity-train --config "$config_path"


## Load Checkpoint and Inspect Weight Variance


In [ ]:
import torch
import pandas as pd
from parity_net.checkpoint import load_checkpoint
from parity_net.train import resolve_device
checkpoint_path = Path(DRIVE_RUN_DIR) / "checkpoints" / "final.pt"
device = resolve_device(config["training"]["device"])
model, payload, _ = load_checkpoint(checkpoint_path, device)
weight_variances = model.weight_variances()
pd.DataFrame([
    {"layer": layer, "variance": variance}
    for layer, variance in weight_variances.items()
])


## PCA Rank-Reduction Analysis
Set `INTERVENTION_LAYER_IDX` and `KEEP_PCS` below. Layer index `0` is the embedded residual stream before the first residual block; later indices are after residual blocks / before the corresponding next block.


In [ ]:
INTERVENTION_LAYER_IDX = 2
KEEP_PCS = 50
PCA_SAMPLES = config["training"]["test_samples"]
ANALYSIS_DIR = str(Path(DRIVE_RUN_DIR) / "analysis")
!parity-analyze   --checkpoint "$checkpoint_path"   --output-dir "$ANALYSIS_DIR"   --pca-samples "$PCA_SAMPLES"   --batch-size "{config['training']['batch_size']}"   --intervention-layer "$INTERVENTION_LAYER_IDX"   --keep-pcs "$KEEP_PCS"


## Results
The first table reports how many dimensions recover 90% and 99% of variance at each layer. The second reports MSE after the PCA intervention, including degree 2, 4, 8, and 16 parity groups.


In [ ]:
rank_df = pd.read_csv(Path(ANALYSIS_DIR) / "pca_rank_thresholds.csv")
intervention_df = pd.read_csv(Path(ANALYSIS_DIR) / "pca_intervention_mse.csv")
baseline_df = pd.read_csv(Path(ANALYSIS_DIR) / "baseline_mse.csv")
print("PCA rank thresholds")
display(rank_df)
print("Baseline MSE")
display(baseline_df)
print("PCA intervention MSE")
display(intervention_df[["intervention_layer", "keep_pcs", "mse_d2", "mse_d4", "mse_d8", "mse_d16", "mse_all"]])
